# Kyrgyzstan Education Statistics — Final Report

**Course:** Statistics Final Project  
**Dataset:** Official National Statistics of the Kyrgyz Republic — Education Sector  
**Period covered:** 1989–2024 (literacy census) · 2010–2024 (students) · 2001–2024 (budget)  

This notebook is the single-source-of-truth report. It runs the full pipeline — loading, cleaning, statistics, visualisations, and interpretations — in one place.

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import linregress, shapiro, spearmanr, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

BLUE   = '#2563EB'
ORANGE = '#EA580C'
GREEN  = '#16A34A'
PURPLE = '#7C3AED'
RED    = '#DC2626'
GRAY   = '#6B7280'

plt.rcParams.update({
    'figure.dpi': 150, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
})

# ── Load ─────────────────────────────────────────────────────────────────────
literacy_raw  = pd.read_csv('literacy.csv',        encoding='utf-8-sig')
cis_raw       = pd.read_csv('students_cis.csv',     encoding='utf-8-sig')
non_cis_raw   = pd.read_csv('students_non_cis.csv', encoding='utf-8-sig')
budget_raw    = pd.read_csv('education_budget.csv', encoding='utf-8-sig')

year_cols = [str(y) for y in range(2010, 2025)]

# ── Literacy ─────────────────────────────────────────────────────────────────
literacy = literacy_raw[['Items','1989','1999','2009','2022']].copy()
literacy.columns = ['Category','1989','1999','2009','2022']
literacy['Category'] = literacy['Category'].str.strip()
for c in ['1989','1999','2009','2022']:
    literacy[c] = pd.to_numeric(literacy[c], errors='coerce')
literacy_clean = literacy[literacy['Category'].notna() & (literacy['Category']!='') & literacy['1989'].notna()].copy()

# ── Students ──────────────────────────────────────────────────────────────────
def extract_by_country(df, group_label):
    en_col = df.columns[2]
    skip = {'total','ncluding:','including:','','nan'}
    rows = []
    for _, row in df.iterrows():
        name = str(row[en_col]).strip()
        if name.lower() in skip or 'Including' in name: continue
        vals = pd.to_numeric(pd.Series([row[y] for y in year_cols]), errors='coerce').values
        rows.append({'Country': name, 'Group': group_label, **dict(zip(year_cols, vals))})
    return pd.DataFrame(rows)

def extract_totals(df):
    en_col = df.columns[2]
    mask = df[en_col].astype(str).str.strip().str.lower().isin(['total'])
    row = df[mask].iloc[0]
    return pd.to_numeric(row[year_cols], errors='coerce')

cis_countries = extract_by_country(cis_raw, 'CIS')
non_cis_countries = extract_by_country(non_cis_raw, 'Non-CIS')
countries_df = pd.concat([cis_countries, non_cis_countries], ignore_index=True)

cis_total = extract_totals(cis_raw)
non_cis_total = extract_totals(non_cis_raw)
students_df = pd.DataFrame({'Year': [int(y) for y in year_cols],
                             'CIS': cis_total.values, 'Non_CIS': non_cis_total.values})
students_df['Total'] = students_df['CIS'] + students_df['Non_CIS']
students_df['CIS_pct'] = students_df['CIS'] / students_df['Total'] * 100
students_df['NonCIS_pct'] = students_df['Non_CIS'] / students_df['Total'] * 100

# ── Budget ────────────────────────────────────────────────────────────────────
budget = budget_raw.copy()
en_col_b = budget.columns[2]
budget[en_col_b] = budget[en_col_b].astype(str).str.strip()
b_years = [str(y) for y in range(2001, 2025)]
for col in b_years:
    if col in budget.columns:
        budget[col] = pd.to_numeric(budget[col], errors='coerce')

def get_budget_row(kw):
    mask = budget[en_col_b].str.lower().str.contains(kw, na=False)
    return budget[mask].iloc[0][b_years].astype(float) if mask.any() else pd.Series([np.nan]*len(b_years), index=b_years)

budget_ts = pd.DataFrame({'Year': [int(y) for y in b_years],
    'Total_mln_som': get_budget_row('total').values,
    'Preschool':     get_budget_row('pre-school').values,
    'Higher':        get_budget_row('higher').values,
})
merged = students_df.merge(budget_ts[budget_ts['Year'] >= 2010][['Year','Total_mln_som']], on='Year')

print('✓ All datasets loaded and cleaned.')
print(f'  Students: {students_df.shape} | Budget: {budget_ts.shape} | Countries: {countries_df.shape}')

---
## 2. Descriptive Statistics

In [ ]:
print('=== 2.1 International Students (2010–2024) ===')
display(students_df[['CIS','Non_CIS','Total']].describe().round(1))

print('\n=== 2.2 Education Budget (mln som) ===')
display(budget_ts[['Total_mln_som','Preschool','Higher']].describe().round(1))

print('\n=== 2.3 Key Growth Metrics ===')
for label, col in [('CIS', 'CIS'), ('Non-CIS', 'Non_CIS'), ('Total', 'Total')]:
    s = students_df[col]
    cagr = (s.iloc[-1] / s.iloc[0]) ** (1/14) - 1
    print(f'  {label:8s} | 2010: {int(s.iloc[0]):>7,} | 2024: {int(s.iloc[-1]):>7,} | '
          f'Peak: {int(s.max()):>7,} ({students_df.loc[s.idxmax(),"Year"]}) | CAGR: {cagr*100:+.1f}%')

b = budget_ts['Total_mln_som'].dropna()
b_cagr = (b.iloc[-1] / b.iloc[0]) ** (1/(len(b)-1)) - 1
print(f'  {"Budget":8s} | 2001: {b.iloc[0]:>10,.1f} | 2024: {b.iloc[-1]:>10,.1f} | CAGR: {b_cagr*100:+.1f}%')

> **Key finding:** Non-CIS student enrollment grew at a CAGR of ~14.8%, far outpacing CIS students at ~6.7%. The education budget grew at a CAGR of ~15.3% annually from 2001–2024 — broadly aligned with student enrollment growth.

---
## 3. Normality & Hypothesis Tests

In [ ]:
print('=== 3.1 Shapiro-Wilk Normality Tests (H₀: data is normally distributed) ===')
test_series = {
    'CIS students':         students_df['CIS'],
    'Non-CIS students':     students_df['Non_CIS'],
    'Total students':       students_df['Total'],
    'Education budget':     budget_ts['Total_mln_som'].dropna(),
    'CIS YoY growth%':      students_df['CIS'].pct_change().dropna()*100,
    'Non-CIS YoY growth%':  students_df['Non_CIS'].pct_change().dropna()*100,
}
rows = []
for name, s in test_series.items():
    w, p = shapiro(s.dropna())
    rows.append({'Series': name, 'W': round(w,4), 'p-value': round(p,4),
                 'Normal (α=0.05)': '✓ Yes' if p > 0.05 else '✗ No'})
display(pd.DataFrame(rows))

print('\n=== 3.2 Mann-Whitney U Test: CIS vs Non-CIS YoY Growth% ===')
cis_growth    = students_df['CIS'].pct_change().dropna() * 100
noncis_growth = students_df['Non_CIS'].pct_change().dropna() * 100
u_stat, u_p = mannwhitneyu(cis_growth, noncis_growth, alternative='two-sided')
print(f'  U = {u_stat:.1f}, p = {u_p:.4f}')
print(f'  Result: {"Reject H₀ — distributions are significantly different (α=0.05)" if u_p < 0.05 else "Fail to reject H₀ — no significant difference"}')

> **Interpretation:** Most raw series are non-normal (p < 0.05 on Shapiro-Wilk), which is expected for monotonically trending time-series data. This motivates the use of Spearman rank correlation alongside Pearson.

---
## 4. Correlation Analysis

In [ ]:
print('=== 4.1 Pearson Correlations ===')
pairs = [
    ('Total students', 'Education budget', 'Total', 'Total_mln_som'),
    ('CIS students',   'Education budget', 'CIS',   'Total_mln_som'),
    ('Non-CIS students','Education budget','Non_CIS','Total_mln_som'),
    ('CIS students',   'Non-CIS students', 'CIS',   'Non_CIS'),
]
for lx, ly, cx, cy in pairs:
    r, p = stats.pearsonr(merged[cx], merged[cy])
    sig = '** Significant' if p < 0.05 else 'Not significant'
    print(f'  {lx:22s} vs {ly:22s} | r = {r:+.4f} | p = {p:.4f} | {sig}')

print('\n=== 4.2 Spearman Correlations (rank-based) ===')
for lx, ly, cx, cy in pairs:
    r, p = spearmanr(merged[cx], merged[cy])
    sig = '** Significant' if p < 0.05 else 'Not significant'
    print(f'  {lx:22s} vs {ly:22s} | ρ = {r:+.4f} | p = {p:.4f} | {sig}')

In [ ]:
# Heatmap
corr_df = merged[['CIS','Non_CIS','Total','Total_mln_som']].copy()
corr_df.columns = ['CIS Students','Non-CIS Students','Total Students','Education Budget']

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Pearson Correlation Matrix (2010–2024)', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('report_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** All student enrollment series correlate strongly and significantly with the education budget (Pearson r > 0.85, p < 0.001). Non-CIS enrollment has the tightest relationship (r ≈ 0.97).

---
## 5. Linear Regression: Budget → Total Students

In [ ]:
X = merged['Total_mln_som'].values
Y = merged['Total'].values
slope, intercept, r_val, p_val, std_err = linregress(X, Y)
Y_pred    = slope * X + intercept
residuals = Y - Y_pred
r2        = r_val**2
adj_r2    = 1 - (1 - r2) * (len(X) - 1) / (len(X) - 2)
rmse      = np.sqrt(np.mean(residuals**2))
mae       = np.mean(np.abs(residuals))

print(f'  Equation:       Total = {slope:.5f} × Budget + {intercept:.2f}')
print(f'  R²:             {r2:.4f}')
print(f'  Adjusted R²:    {adj_r2:.4f}')
print(f'  p-value:        {p_val:.6f}')
print(f'  Std Error:      {std_err:.5f}')
print(f'  RMSE:           {rmse:,.0f} students')
print(f'  MAE:            {mae:,.0f} students')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter + regression line
ax = axes[0]
ax.scatter(X, Y, color=BLUE, s=70, zorder=5, label='Observed')
for i, row in merged.iterrows():
    ax.annotate(str(int(row['Year'])), (row['Total_mln_som'], row['Total']),
                fontsize=8, xytext=(4,4), textcoords='offset points', color=GRAY)
x_line = np.linspace(X.min(), X.max(), 100)
ax.plot(x_line, slope*x_line + intercept, color=RED, linewidth=2,
        label=f'OLS  R² = {r2:.4f}')
ax.set_title('Budget vs Total Students (OLS)', fontweight='bold')
ax.set_xlabel('Education Budget (mln som)')
ax.set_ylabel('Total International Students')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(framealpha=0.3)

# Residuals
ax2 = axes[1]
ax2.scatter(Y_pred, residuals, color=ORANGE, s=60, alpha=0.8)
ax2.axhline(0, color=RED, linestyle='--', linewidth=1.5)
ax2.set_xlabel('Fitted Values')
ax2.set_ylabel('Residuals')
ax2.set_title('Residuals vs Fitted', fontweight='bold')

plt.tight_layout()
plt.savefig('report_regression.png', dpi=150, bbox_inches='tight')
plt.show()

> **Finding:** The linear model explains ~85% of variance in total enrollment. The residual plot shows a non-random pattern (positive residuals in 2020–2021), suggesting the COVID spike is not captured by a simple linear model and that a more flexible model is warranted for forecasting.

---
## 6. Key Visualisations

In [ ]:
# International Students Time Series
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.stackplot(students_df['Year'], students_df['CIS'], students_df['Non_CIS'],
             labels=['CIS','Non-CIS'], colors=[BLUE, ORANGE], alpha=0.80)
ax.set_title('International Students — Stacked Total', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Students')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(loc='upper left', framealpha=0.3)

ax2 = axes[1]
ax2.plot(students_df['Year'], students_df['CIS_pct'],    'o-', color=BLUE,   linewidth=2, label='CIS %')
ax2.plot(students_df['Year'], students_df['NonCIS_pct'], 's-', color=ORANGE, linewidth=2, label='Non-CIS %')
ax2.axhline(50, color=GRAY, linestyle='--', linewidth=0.8)
ax2.set_title('CIS vs Non-CIS Share Over Time', fontweight='bold')
ax2.set_xlabel('Year'); ax2.set_ylabel('%'); ax2.legend(framealpha=0.3)
ax2.set_xticks(students_df['Year']); ax2.set_xticklabels(students_df['Year'], rotation=45)

plt.tight_layout()
plt.savefig('report_students_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Budget trend
fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(budget_ts['Year'], budget_ts['Total_mln_som'], alpha=0.15, color=BLUE)
ax.plot(budget_ts['Year'], budget_ts['Total_mln_som'], 'o-', color=BLUE, linewidth=2.5, markersize=5, label='Total')
ax.plot(budget_ts['Year'], budget_ts['Preschool'],     '^-', color=GREEN,  linewidth=2, label='Pre-school')
ax.plot(budget_ts['Year'], budget_ts['Higher'],        's-', color=ORANGE, linewidth=2, label='Higher Education')
ax.set_title('Education Budget Trends 2001–2024 (mln som)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Year'); ax.set_ylabel('mln som')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.legend(framealpha=0.3)
plt.tight_layout()
plt.savefig('report_budget_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Donut — 2024 country shares
countries_2024 = countries_df[['Country','Group','2024']].copy()
countries_2024['2024'] = pd.to_numeric(countries_2024['2024'], errors='coerce').fillna(0)
countries_2024 = countries_2024[countries_2024['2024'] > 0].sort_values('2024', ascending=False)

main = countries_2024[countries_2024['2024'] >= 300].copy()
small = countries_2024[countries_2024['2024'] < 300]
if len(small) > 0:
    other_row = pd.DataFrame([{'Country': 'Others', 'Group': 'Mixed', '2024': small['2024'].sum()}])
    main = pd.concat([main, other_row], ignore_index=True)

palette_c  = ['#1d4ed8','#2563eb','#3b82f6','#93c5fd']
palette_nc = ['#c2410c','#ea580c','#f97316','#fdba74']
ci, nc2 = 0, 0
colors_d = []
for _, row in main.iterrows():
    if row['Group'] == 'CIS':
        colors_d.append(palette_c[ci % len(palette_c)]); ci += 1
    else:
        colors_d.append(palette_nc[nc2 % len(palette_nc)]); nc2 += 1

fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    main['2024'], labels=main['Country'], autopct='%1.1f%%',
    colors=colors_d, startangle=140, pctdistance=0.82,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=1.5),
    textprops={'fontsize': 9}
)
for at in autotexts: at.set_fontsize(8)
ax.text(0, 0, f'{int(main["2024"].sum()):,}\nstudents\n2024', ha='center', va='center',
        fontsize=12, fontweight='bold', color='#1e293b')
cis_p = mpatches.Patch(color=palette_c[0],  label='CIS')
nc_p  = mpatches.Patch(color=palette_nc[0], label='Non-CIS')
ax.legend(handles=[cis_p, nc_p], loc='lower right', framealpha=0.3)
ax.set_title('Country Share of International Students — 2024', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('report_donut_2024.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Summary of Findings

| Dimension | Key Finding |
|---|---|
| **Literacy** | Higher education attainment nearly tripled from 1989–2022; illiteracy declined by ~75% over the same period |
| **CIS Students** | Peaked at 57,103 in 2021 (COVID anomaly); declining post-2021 to 25,073 in 2024 |
| **Non-CIS Students** | Grew continuously from 3,366 (2010) to 24,480 (2024); now approaching parity with CIS |
| **Dominant origin** | Uzbekistan, India, and Pakistan account for ~82% of all international students in 2024 |
| **Budget** | Grew from 2,848 mln som (2001) to 83,924 mln som (2024), a 29× increase; CAGR ≈ 15.3% |
| **Correlation** | Strong positive correlation between budget and enrollment (Pearson r ≈ 0.87–0.97) |
| **Regression** | Linear model: R² ≈ 0.85; non-random residuals suggest the COVID spike is an outlier |

## 8. Conclusions

1. Kyrgyzstan's education sector has undergone substantial transformation since independence — rising attainment, increasing international student intake, and steadily growing public investment.
2. The sharp 2019–2021 enrollment surge (particularly CIS) appears linked to post-pandemic policy changes and regional factors rather than organic growth, and has since partially reversed.
3. Non-CIS student growth is structurally more stable and is now the primary driver of international enrollment. South Asian students (India, Pakistan) dominate this segment.
4. Budget and enrollment move together — the correlation is strong and significant — but causality is unclear: rising enrollment could be pulling budget allocations, or vice versa.
5. Forecasting (see `kyrgyzstan_forecasting.ipynb`) suggests continued moderate growth in both budget and enrollment through 2030, with ARIMA projections being the most conservative.